# Train a tiny resume parser on eight pages

This notebook shows **how training works**; it does not produce a useful model. To read CVs with the released
models, open [try_resuvl.ipynb](try_resuvl.ipynb) instead (about 5 minutes, no GPU needed).

This notebook trains a ResuVL-S model, the smallest of the three, on the six training pages in
`examples/`, then scores it on those pages and on two pages it has not seen. It runs the same code as the full
training, so every step of the pipeline can be watched and changed.

**Where to run it.** On Colab, choose *Runtime > Change runtime type > T4 GPU*; the whole notebook took 16 minutes
in a test run on a T4. On a local machine, install with
`pip install -e ".[notebook]"` and open the notebook from the repository; it uses a CUDA GPU or Apple MPS if one is
present and otherwise the CPU (the whole notebook took 8 minutes on an Apple M2 Max).

**What to expect.** Six pages teach the model the output format and let it memorise those pages; they do not teach
it to read new CVs. The full recipe in `docs/training.md` uses 1,393 pages.

Outputs go to `runs/example/` in the repository folder.

## 1. Setup

On Colab this clones the repository and installs the package (about 1 minute). Locally it only moves to the
repository root. `TINY_RESUME_PARSER_DATA` points the package at `examples/` and must be set before the package is
imported.

In [ ]:
import logging
import os
import sys
import warnings
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("/content/tiny-resume-parser").exists():
        !git clone -q https://github.com/iamkucuk/tiny-resume-parser /content/tiny-resume-parser
    %cd /content/tiny-resume-parser
    %pip install -q -e .
elif Path.cwd().name == "examples":
    os.chdir("..")

os.environ["TINY_RESUME_PARSER_DATA"] = "examples"
sys.path.insert(0, "src")  # an editable install is only seen by a restarted kernel

# Two known warnings that do not apply here: a tokenizer regex notice meant for another model family, and
# pin_memory on Apple GPUs. Every other warning still shows.
warnings.filterwarnings("ignore", message=".*pin_memory.*")
logging.getLogger("transformers.tokenization_utils_base").addFilter(lambda r: "fix_mistral_regex" not in r.getMessage())

MODEL = "ibm-granite/granite-docling-258M"
OUT = Path("runs/example")
# The ResuVL-S recipe (configs/resuvl-s.toml): lr 1e-4, 12 image tiles, image noise on half the
# pages. Different here: 15 epochs and one optimizer step per page (--accum 1), because there are only six pages.
EPOCHS = 15
print(f"working in {Path.cwd()}, outputs to {OUT}")

## 2. One page and its label

Each line of `examples/train.jsonl` holds a page image path and the record the teacher model wrote for it. The
student learns to write the compact form below: the record as JSON with empty fields left out.

In [ ]:
from tiny_resume_parser.data import page_image, read_jsonl
from tiny_resume_parser.models import student_target

train_rows = read_jsonl("examples/train.jsonl")
eval_rows = read_jsonl("examples/eval.jsonl")
row = train_rows[0]
display(page_image(row).resize((496, 702)))
target = student_target(row["record"])
print(target[:1000], "...")
print(f"{len(train_rows)} training pages, {len(eval_rows)} unseen pages; this target is {len(target)} characters")

## 3. Train

This runs `tiny-resume-parser train` in the notebook process so the progress bar shows here. Before the first step
it prints a mask check: the number of image tokens (none of them learnt) and of answer tokens (the only ones in the
loss). On a T4 the run uses float32 because the T4 has no bf16. The batch is already one page; if it runs out of
memory, restart the runtime and close other notebooks.

In [ ]:
from tiny_resume_parser import train

args = f"--model {MODEL} --lr 1e-4 --max-patches 12 --augment 0.5 --epochs {EPOCHS} --accum 1 --logging-steps 5"
train.main(f"{args} --train-file examples/train.jsonl --out {OUT}".split())

## 4. Training loss

The loss is the mean cross-entropy over the answer tokens. On six pages it falls from about 1.5 to below 0.01: the model
has memorised them.

In [ ]:
import json

import matplotlib.pyplot as plt

log = [e for e in json.loads((OUT / "log_history.json").read_text()) if "loss" in e]
plt.figure(figsize=(6, 3))
plt.plot([e["step"] for e in log], [e["loss"] for e in log])
plt.xlabel("optimizer step (one page)")
plt.ylabel("training loss")
plt.title("ResuVL-S on six pages")
plt.show()
print(f"loss {log[0]['loss']:.2f} at step {log[0]['step']}, {log[-1]['loss']:.3f} at step {log[-1]['step']}")

## 5. Score on seen and unseen pages

`evaluate` reads each page, compares the answer with the label field by field and writes one line per page to
`pages.jsonl` (docs/evaluation.md explains the score). In two test runs: field F1 0.78 and 1.00 on the six training pages,
0.16 and 0.01 on the two unseen pages. With six pages, the random image noise makes each run differ. A page takes 20 to 40 seconds on a T4 (float32; the T4 has no bf16) and 6
to 36 seconds on an Apple M2 Max; an answer that repeats itself runs to the 2,500-token limit and takes longest.

Loading the saved model prints a warning about an "incorrect regex pattern" in the tokenizer. It does not apply
here: the saved and the original tokenizer give identical token ids on all eight labels.

In [ ]:
from tiny_resume_parser import evaluate

seen = evaluate.main(f"--model {OUT}/model --labels examples/train.jsonl --out {OUT}/eval_train".split())
unseen = evaluate.main(f"--model {OUT}/model --labels examples/eval.jsonl --out {OUT}/eval_unseen".split())
print(f"field F1 (strict): seen pages {seen['field_f1']:.2f}, unseen pages {unseen['field_f1']:.2f}")

## 6. Read an unseen answer

The answer below is for a page the model never saw. Its structure is right; the values show what six pages are not
enough for.

In [ ]:
page = json.loads((OUT / "eval_unseen" / "pages.jsonl").read_text().splitlines()[0])
display(page_image(eval_rows[0]).resize((496, 702)))
print(page["status"], "\n", page["answer"][:1500])

## 7. Repair the broken answers

An unparsed answer scores 0 even when its first fields are right. `rescore --repair` cuts a broken answer at the
point where the JSON breaks and keeps the complete values before it, as `tiny-resume-parser parse` does for real
use. Repair raises the unseen pages' score only a little, because most label values are never written. The full models in the README were
trained on 1,393 pages and score 0.65 to 0.85 lenient F1 on 204 test pages.

In [ ]:
from tiny_resume_parser import rescore

rescore.main(f"--repair --labels examples/eval.jsonl {OUT}/eval_unseen".split())

## 8. Explore one page

The explorer reads one page with the trained model and shows why it gets the score it gets. Pick an example page
(or upload a page image or PDF; the first page is read) and press **Read page**.

- **Fields**: every (field, value) pair of the answer next to the label's value. *match*: same letters and digits;
  *differs*: same field, but a letter or digit differs (highlighted); *extra*: not in the label; *missing*: in the
  label but left out. Field F1 is computed from exactly these pairs.
- **Scores**: strict, lenient and content F1 for this page, each with its rule.
- **JSON**: the answer after repair, as `tiny-resume-parser parse` returns it.
- **Model input**: the image tiles the model receives and how many tokens they become.

It starts on the first unseen page. Each read takes 20 to 40 seconds on a T4.

In [ ]:
sys.path.insert(0, "examples")
from explorer import Explorer, report

explorer = Explorer(OUT / "model", train_rows, eval_rows)
explorer.show()

Widgets do not render in a saved copy of a notebook (for example on GitHub). The next cell prints the scores and the
field table of the page shown above as plain HTML, so a saved copy still shows them.

In [ ]:
report(explorer)

## 9. Compare with a released model

The same two unseen pages, read by the released ResuVL-S: the same model size and recipe, trained on 1,393 pages
instead of six. The gap between the two scores is what the extra pages buy. The first run downloads the model; a
page takes about as long as in step 5.

In [ ]:
released = evaluate.main(f"--model furkankucuk/ResuVL-S --labels examples/eval.jsonl --out {OUT}/eval_released".split())
print(f"field F1 (strict) on the two unseen pages: six-page model {unseen['field_f1']:.2f}, ResuVL-S {released['field_f1']:.2f}")

## 10. Release the GPU (Colab)

Colab GPU time is limited; this ends the session. Copy anything from `runs/example/` first.

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import runtime

    runtime.unassign()